## Policy Training
Optional Dueling Double DQN experiment using the existing trainer; transition data is not supplied.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
import tomllib

TRAIN_POLICY = False
policy = tomllib.loads((ROOT / "configs/policy.toml").read_text())
policy


### Transitions
Inspect episode-level inputs before enabling training; this policy is separate from the default recommender.

In [ ]:
from hlo_recommender.models.dueling import validate_transitions

available = {}
for name in ("train", "validation"):
    path = ROOT / policy[name]
    available[name] = path.is_file()
    if path.is_file():
        validate_transitions(read_jsonl(path), policy["state_dim"], policy["action_count"])
available


In [ ]:
if TRAIN_POLICY:
    OUTPUT = ROOT / "artifacts/notebooks" / f"policy-{uuid4().hex[:8]}"
    for key in ("train", "validation"):
        policy[key] = str(ROOT / policy[key])
    policy["output"] = str(OUTPUT)
    CONFIG_PATH = ROOT / "artifacts/notebooks/configs" / f"{OUTPUT.name}.toml"
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text("\n".join(f"{key} = {json.dumps(value)}" for key, value in policy.items()) + "\n")
    result = run(parser().parse_args(["train", "policy", "--config", str(CONFIG_PATH)]))
    print(json.dumps(result, indent=2))
else:
    print("Policy training was not executed.")
